In [7]:
import sys
from collections import defaultdict
import heapq
from itertools import combinations
import math

class Assignment:
    def __init__(self, a_id, in1, in2, out, food):
        self.a_id = a_id
        self.in1 = in1
        self.in2 = in2
        self.out = out
        self.food = food

class Scheduler:
    def __init__(self, data_string):
        self.food_costs = {}
        self.group_size = 0
        self.initial_items = set()
        self.goals = set()
        self.assignments = {}
        self.parse_data(data_string)
        self.compute_metrics()

    def parse_data(self, data_string):
        lines = [line.strip() for line in data_string.strip().split('\n') if line.strip()]
        for line in lines:
            parts = line.split()
            if not parts: continue

            if parts[0] == 'C' and len(parts) == 3:
                self.food_costs[parts[1]] = int(parts[2])
            elif parts[0] == 'G':
                self.group_size = int(parts[1])
            elif parts[0] == 'I':
                self.initial_items = set(int(x) for x in parts[1:] if x != '-1')
            elif parts[0] == 'O':
                self.goals = set(int(x) for x in parts[1:] if x != '-1')
            elif parts[0] == 'A':
                a_id = int(parts[1])
                self.assignments[a_id] = Assignment(
                    a_id, int(parts[2]), int(parts[3]), int(parts[4]), parts[5]
                )

    def compute_metrics(self):
        # 1. Depths (longest path to a leaf - used for Critical Path)
        self.depths = {a_id: 1 for a_id in self.assignments}
        changed = True
        while changed:
            changed = False
            for a_id, assign in self.assignments.items():
                for other_id, other_assign in self.assignments.items():
                    if assign.out in (other_assign.in1, other_assign.in2):
                        if self.depths[a_id] < self.depths[other_id] + 1:
                            self.depths[a_id] = self.depths[other_id] + 1
                            changed = True

        # 2. Levels (longest path from initial inputs - used for Earliest Deadline)
        self.levels = {a_id: 1 for a_id in self.assignments}
        changed = True
        while changed:
            changed = False
            for a_id, assign in self.assignments.items():
                in1_lvl, in2_lvl = 0, 0
                for other_id, other_assign in self.assignments.items():
                    if other_assign.out == assign.in1:
                        in1_lvl = max(in1_lvl, self.levels[other_id])
                    if other_assign.out == assign.in2:
                        in2_lvl = max(in2_lvl, self.levels[other_id])
                new_lvl = max(in1_lvl, in2_lvl) + 1
                if self.levels[a_id] < new_lvl:
                    self.levels[a_id] = new_lvl
                    changed = True

    def get_ready_assignments(self, available_items, completed):
        ready = []
        for a_id, assign in self.assignments.items():
            if a_id not in completed:
                if assign.in1 in available_items and assign.in2 in available_items:
                    ready.append(a_id)
        return ready

    def run_all_strategies(self):
        greedy_results = {}
        strategies = ["depth", "cost", "frequency", "earliest"]

        print("--- TASK 1: GREEDY APPROACHES ---")
        for strat in strategies:
            sched = self.greedy_schedule(strat)
            days, cost = self.print_schedule(f"Greedy by {strat.capitalize()}", sched)
            greedy_results[strat] = {"days": days, "cost": cost}

        # Find best greedy (Primary: min days, Secondary: min cost)
        best_greedy = min(greedy_results.values(), key=lambda x: (x['days'], x['cost']))

        print("--- TASK 2: A* SEARCH ---")
        sched_astar, states = self.a_star_search()
        astar_days, astar_cost = self.print_schedule("A* Optimal Schedule", sched_astar, states)

        print("--- COMPARISON ---")
        day_diff = best_greedy['days'] - astar_days
        cost_diff = best_greedy['cost'] - astar_cost
        print(f"Vs Best Greedy -> Day Difference: {day_diff} (A* saved {day_diff} days)")
        print(f"Vs Best Greedy -> Cost Difference: {cost_diff} (A* saved {cost_diff} cost)")


    def print_schedule(self, title, schedule, total_states=None):
        print(f"\nStrategy: {title}")
        total_cost = 0
        for day, tasks in enumerate(schedule, 1):
            print(f"  Day-{day}: {', '.join(['A'+str(t) for t in tasks])}")

            menu_counts = defaultdict(int)
            day_cost = 0
            for t in tasks:
                food = self.assignments[t].food
                menu_counts[food] += 1
                day_cost += self.food_costs[food]

            menu_str = ", ".join([f"{count}-{food}" for food, count in menu_counts.items()])
            print(f"    Menu: {menu_str}")
            print(f"    Cost: {day_cost}")
            total_cost += day_cost

        print(f"  Total Days: {len(schedule)}")
        print(f"  Total Cost: {total_cost}")
        if total_states is not None:
            print(f"  States Explored: {total_states}")
        return len(schedule), total_cost

    def greedy_schedule(self, strategy):
        available = set(self.initial_items)
        completed = set()
        schedule = []

        while len(completed) < len(self.assignments):
            ready = self.get_ready_assignments(available, completed)
            if not ready:
                print("Error: Deadlock detected.")
                return []

            if strategy == "depth":
                ready.sort(key=lambda x: (-self.depths[x], x))
            elif strategy == "cost":
                ready.sort(key=lambda x: (self.food_costs[self.assignments[x].food], x))
            elif strategy == "frequency":
                freq = defaultdict(int)
                for a_id in self.assignments:
                    if a_id not in completed:
                        freq[self.assignments[a_id].food] += 1
                ready.sort(key=lambda x: (-freq[self.assignments[x].food], x))
            elif strategy == "earliest":
                ready.sort(key=lambda x: (self.levels[x], x))

            selected = ready[:self.group_size]
            schedule.append(selected)
            completed.update(selected)

            for t in selected:
                available.add(self.assignments[t].out)

        return schedule

    def a_star_search(self):
        start_avail = frozenset(self.initial_items)
        start_comp = frozenset()

        def h(comp):
            return math.ceil((len(self.assignments) - len(comp)) / self.group_size)

        pq = []
        heapq.heappush(pq, (h(start_comp), 0, start_comp, start_avail, []))
        visited = set()
        states_explored = 0

        while pq:
            f, g, comp, avail, sched = heapq.heappop(pq)
            states_explored += 1

            if len(comp) == len(self.assignments):
                return sched, states_explored

            state_key = comp
            if state_key in visited:
                continue
            visited.add(state_key)

            ready = self.get_ready_assignments(avail, comp)

            max_tasks = min(self.group_size, len(ready))
            valid_moves = []
            for i in range(max_tasks, 0, -1):
                valid_moves.extend(list(combinations(ready, i)))
                if valid_moves:
                    break

            for move in valid_moves:
                new_comp = set(comp)
                new_comp.update(move)
                new_avail = set(avail)
                for t in move:
                    new_avail.add(self.assignments[t].out)

                new_sched = sched + [list(move)]
                new_g = g + 1
                new_f = new_g + h(new_comp)

                heapq.heappush(pq, (new_f, new_g, frozenset(new_comp), frozenset(new_avail), new_sched))

        return [], states_explored


# ==========================================
# TEST CASES
# ==========================================

tc1 = """
C TC 1
C DF 1
C PM 1
C GJ 1
G 3
I 1 2 3 4 5 6 -1
O 13 14 17 -1
A 1 1 3 7 TC
A 2 4 2 8 TC
A 3 1 3 9 TC
A 4 2 3 10 PM
A 5 7 8 11 TC
A 6 4 6 12 TC
A 7 6 9 13 PM
A 8 10 5 14 GJ
A 9 1 11 15 DF
A 10 3 12 16 TC
A 11 15 16 17 DF
"""

tc2 = """
C TC 1
C DF 2
C PM 8
C GJ 3
G 2
I 1 -1
O 10 11 12 -1
A 1 1 1 2 PM
A 2 1 1 3 TC
A 3 1 1 4 TC
A 4 1 1 5 TC
A 5 2 2 6 DF
A 6 6 6 7 DF
A 7 7 7 8 DF
A 8 8 8 10 DF
A 9 3 4 11 GJ
A 10 5 5 12 GJ
"""

tc3 = """
C TC 2
C DF 2
C PM 4
C GJ 6
G 3
I 1 -1
O 20 6 7 8 9 10 11 -1
A 1 1 1 2 PM
A 2 2 2 3 PM
A 3 3 3 4 PM
A 4 4 4 5 PM
A 5 5 5 20 PM
A 6 1 1 6 DF
A 7 1 1 7 DF
A 8 1 1 8 DF
A 9 1 1 9 DF
A 10 1 1 10 DF
A 11 1 1 11 DF
"""

if __name__ == "__main__":
    test_cases = [("Test Case 1 (Sample from PDF)", tc1),
                  ("Test Case 2 (Bottleneck vs Cheap Food)", tc2),
                  ("Test Case 3 (Deep Chain vs High Frequency Food)", tc3)]

    for name, data in test_cases:
        print(f"\n{'='*60}")
        print(f"RUNNING: {name}")
        print(f"{'='*60}\n")

        scheduler = Scheduler(data)
        scheduler.run_all_strategies()


RUNNING: Test Case 1 (Sample from PDF)

--- TASK 1: GREEDY APPROACHES ---

Strategy: Greedy by Depth
  Day-1: A1, A2, A6
    Menu: 3-TC
    Cost: 3
  Day-2: A5, A3, A4
    Menu: 2-TC, 1-PM
    Cost: 3
  Day-3: A9, A10, A7
    Menu: 1-DF, 1-TC, 1-PM
    Cost: 3
  Day-4: A8, A11
    Menu: 1-GJ, 1-DF
    Cost: 2
  Total Days: 4
  Total Cost: 11

Strategy: Greedy by Cost
  Day-1: A1, A2, A3
    Menu: 3-TC
    Cost: 3
  Day-2: A4, A5, A6
    Menu: 1-PM, 2-TC
    Cost: 3
  Day-3: A7, A8, A9
    Menu: 1-PM, 1-GJ, 1-DF
    Cost: 3
  Day-4: A10
    Menu: 1-TC
    Cost: 1
  Day-5: A11
    Menu: 1-DF
    Cost: 1
  Total Days: 5
  Total Cost: 11

Strategy: Greedy by Frequency
  Day-1: A1, A2, A3
    Menu: 3-TC
    Cost: 3
  Day-2: A5, A6, A4
    Menu: 2-TC, 1-PM
    Cost: 3
  Day-3: A9, A7, A8
    Menu: 1-DF, 1-PM, 1-GJ
    Cost: 3
  Day-4: A10
    Menu: 1-TC
    Cost: 1
  Day-5: A11
    Menu: 1-DF
    Cost: 1
  Total Days: 5
  Total Cost: 11

Strategy: Greedy by Earliest
  Day-1: A1, A2, A3
    